In [5]:
import io
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ==========================================
# 1. LOAD DATASET
# ==========================================
# Replace the string below with your file parsing logic, e.g.:
# df = pd.read_csv('student-mat.csv', sep=';')

# Reading raw data provided from the dataset context
raw_data = """school;sex;age;address;famsize;Pstatus;Medu;Fedu;Mjob;Fjob;reason;guardian;traveltime;studytime;failures;schoolsup;famsup;paid;activities;nursery;higher;internet;romantic;famrel;freetime;goout;Dalc;Walc;health;absences;G1;G2;G3
"GP";"F";18;"U";"GT3";"A";4;4;"at_home";"teacher";"course";"mother";2;2;0;"yes";"no";"no";"no";"yes";"yes";"no";"no";4;3;4;1;1;3;4;"0";"11";11
"GP";"F";17;"U";"GT3";"T";1;1;"at_home";"other";"course";"father";1;2;0;"no";"yes";"no";"no";"no";"yes";"yes";"no";5;3;3;1;1;3;2;"9";"11";11
"GP";"F";15;"U";"LE3";"T";1;1;"at_home";"other";"other";"mother";1;2;0;"yes";"no";"no";"no";"yes";"yes";"yes";"no";4;3;2;2;3;3;6;"12";"13";12
""" # (... truncated placeholder for local loading)

# For demonstration, let's parse the string structure directly:
df = pd.read_csv(io.StringIO(raw_data), sep=';')

# ==========================================
# 2. DEFINE TARGET & DROP LEAKAGE FEATURES
# ==========================================
# According to standard evaluation, a grade >= 10 out of 20 is a PASS (1), else FAIL (0)
df['pass'] = (df['G3'] >= 10).astype(int)

# CRITICAL STRATEGY: Drop G1, G2, and G3 to prevent label leakage.
# If you want to include mid-year grades for predicting final outcome, remove 'G1' and 'G2' from this list.
features_to_drop = ['G1', 'G2', 'G3', 'pass']
X = df.drop(columns=features_to_drop)
y = df['pass']

# ==========================================
# 3. PREPROCESSING & ENCODING
# ==========================================
# Convert all binary string objects ("yes"/"no", "GP"/"MS") and nominal features into dummy variables
# drop_first=True prevents the multicollinearity dummy variable trap!
X = pd.get_dummies(X, drop_first=True)

# Ensure all boolean dummy indicators are numeric 1s and 0s
X = X.astype(float)

# ==========================================
# 4. TRAIN-TEST SPLIT
# ==========================================
# Stratify ensures the pass/fail proportions remain identical in both splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ==========================================
# 5. FEATURE SCALING (MANDATORY FOR SVM)
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 6. HYPERPARAMETER TUNING & TRAINING
# ==========================================
# probability=True allows calculation of prediction probabilities for AUC-ROC metric profiles
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_svm = grid_search.best_estimator_
print(f"Optimal Hyperparameters: {grid_search.best_params_}\n")

# ==========================================
# 7. MODEL EVALUATION
# ==========================================
y_pred = best_svm.predict(X_test_scaled)
y_prob = best_svm.predict_proba(X_test_scaled)[:, 1]

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Performance ---")
print(classification_report(y_test, y_pred))

try:
    print(f"AUC-ROC Score: {roc_auc_score(y_test, y_prob):.4f}")
except ValueError:
    print("AUC-ROC score skipped (requires representation of both classes in sample array split)")


ValueError: Cannot have number of splits n_splits=5 greater than the number of samples: n_samples=2.